[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/InterstellarNerd8760/lunar-atlas/blob/proxima/terrain-notebooks/notebooks/prepare-data-batch.ipynb)

# Lunar Atlas dataset scripts

Open this notebook in Google Colab using a **CPU** runtime. Run **Setup** once, then run only the dataset cell you want. Each cell downloads its own ZIP when finished. Run Apollo first for a fast preview. The full normalized WAC mosaic needs about 1.4 GB of source rasters (eight 84 MB equatorial tiles and two 331 MB polar tiles).

Source rasters stay in the Colab session cache at `/content/lunar-atlas-data/cache`. If a source URL fails, open the dataset's product page, download the named raster, and upload it into that cache folder before rerunning. Export sidecars preserve source URLs and georeferencing.

**Viewer compatibility.** Extract `wac-texture.jpg` from the WAC ZIP and place it at `public/data/wac-texture.jpg` in your Lunar Atlas checkout. Keep the sidecar alongside it. The Apollo PNG, polar DEM, horizon profile, and visibility overlays are prepared inputs for viewer integration. PR #2 needs a loader for each of those new formats.


## Setup

In [ ]:
%pip -q install rasterio scipy pillow requests


In [ ]:
from pathlib import Path
import json
import shutil
from datetime import datetime, timezone
from urllib.parse import urlparse

import numpy as np
import requests
import rasterio as rio
from rasterio.enums import Resampling
from rasterio.transform import from_bounds
from rasterio.warp import reproject
from PIL import Image
from IPython.display import display
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from google.colab import files

ROOT = Path('/content/lunar-atlas-data')
CACHE = ROOT / 'cache'
CACHE.mkdir(parents=True, exist_ok=True)
RADIUS = 1_737_400.0
GEO = rio.crs.CRS.from_proj4(f'+proj=longlat +R={RADIUS} +no_defs')
LROC = 'https://pds.lroc.im-ldi.com/data/LRO-L-LROC-5-RDR-V1.0/LROLRC_2001'
session = requests.Session()
session.headers['User-Agent'] = 'LunarAtlasColab/1.0 (https://github.com/InterstellarNerd8760/lunar-atlas)'
session.mount('https://', HTTPAdapter(max_retries=Retry(
    total=3, backoff_factor=1, status_forcelist=[429, 500, 502, 503, 504])))

def fetch(url):
    name = Path(urlparse(url).path).name
    path = CACHE / name
    if path.exists():
        return path
    print('Downloading', name, flush=True)
    partial = path.with_name('partial-' + path.name)
    try:
        with session.get(url, stream=True, timeout=(20, 180)) as response:
            response.raise_for_status()
            with partial.open('wb') as output:
                for chunk in response.iter_content(1024 * 1024):
                    output.write(chunk)
        # Catch redirects to HTML pages and invalid raster payloads before caching.
        with rio.open(partial) as source:
            print(source.width, '×', source.height)
        partial.replace(path)
    except Exception as exc:
        partial.unlink(missing_ok=True)
        raise RuntimeError(
            f'Could not read {url}. If you downloaded the original raster manually, '
            f'put it in {CACHE} as {name} and rerun.') from exc
    return path

def output_dir(name):
    path = ROOT / name
    path.mkdir(exist_ok=True)
    return path

def sidecar(path, **metadata):
    path.write_text(json.dumps(dict(processedUtc=datetime.now(timezone.utc).isoformat(),
                                    **metadata), indent=2, allow_nan=False))

def physical(source, shape=None, resampling=Resampling.average):
    options = {} if shape is None else dict(out_shape=shape, resampling=resampling)
    data = source.read(1, masked=True, out_dtype='float32', **options).filled(np.nan)
    return data * source.scales[0] + source.offsets[0]

def image(data, low, high, alpha=False):
    valid = np.isfinite(data)
    if not valid.any() or high <= low:
        raise ValueError('Raster has no usable display range')
    gray = np.rint(255 * np.clip((np.where(valid, data, low) - low) / (high-low), 0, 1)).astype('uint8')
    if alpha:
        return Image.fromarray(np.stack([gray, valid.astype('uint8') * 255], axis=-1), 'LA')
    return Image.fromarray(gray)

def pack(out):
    zipped = shutil.make_archive(str(out), 'zip', out)
    files.download(zipped)


## 1. Apollo 11 texture

Export a 4 km crop around Tranquility Base at 5 m per pixel. Source: [LROC controlled NAC mosaic](https://data.lroc.im-ldi.com/lroc/view_rdr_product/NAC_ROI_APOLLO11HIB_E006N0235_5M), using the 8-bit browse GeoTIFF of the 5 m product. The mosaic itself is only about 2.8 km wide, so the east and west edges of the 4 km crop will be transparent.


In [ ]:
name = 'NAC_ROI_APOLLO11HIB_E006N0235_5M'
url = f'{LROC}/EXTRAS/BROWSE/NAC_ROI/APOLLO11HIB/{name}.TIF'
out = output_dir('apollo11-nac')
lon, lat = 23.47314, 0.67416
half_width_m = 2000
grid = rio.crs.CRS.from_proj4(
    f'+proj=aeqd +lat_0={lat} +lon_0={lon} +R={RADIUS} +units=m +no_defs')
bounds = [-half_width_m, -half_width_m, half_width_m, half_width_m]
size = 800  # A 4 km crop at the native 5 m sampling.
transform = from_bounds(*bounds, size, size)
crop = np.full((size, size), np.nan, dtype='float32')
with rio.open(fetch(url)) as src:
    if not src.crs:
        raise ValueError('The controlled mosaic has no readable projection')
    reproject(physical(src), crop, src_transform=src.transform, src_crs=src.crs,
              dst_transform=transform, dst_crs=grid, src_nodata=np.nan,
              dst_nodata=np.nan, resampling=Resampling.bilinear)
if not np.isfinite(crop[size//2, size//2]):
    raise ValueError('Source mosaic did not cover the landing center')
lo, hi = np.nanpercentile(crop, [2, 98])
preview = image(crop, lo, hi, alpha=True)
preview.save(out / 'apollo-11-texture.png')
sidecar(out / 'apollo-11-texture.json', source=url, credit='NASA/GSFC/Arizona State University, LROC NAC',
        site='Apollo 11', centerLatLon=[lat, lon], extentM=bounds, width=size, height=size,
        crsWkt=grid.to_wkt(), affine=list(transform)[:6], resolutionM=5,
        displayStretch=[float(lo), float(hi)], rowOrder='north to south',
        note='High-Sun mosaic with source lighting. Transparent pixels are outside valid coverage.')
display(preview.copy().resize((400, 400)))
pack(out)


## 2. South pole elevation and terrain horizon

Export a compact 1024 × 1024 overview from the [80 m LOLA DEM](https://pgda.gsfc.nasa.gov/products/90), and compute one terrain horizon from the native raster. The editable starting point is an illustrative point near Shackleton, with a two-meter observer height. The profile assumes a 150 km search radius and does not resolve terrain below the source grid spacing.

In [ ]:
url = 'https://pgda.gsfc.nasa.gov/data/LOLA_20mpp/LDEM_80S_80MPP_ADJ.TIF'
out = output_dir('south-pole-lola')
# Demonstration point near the Shackleton rim. Replace with your desired site coordinates.
lon, lat, eye_m = 120.0, -89.57, 2.0
max_range_m = 150_000
azimuth = np.arange(360, dtype=float)
with rio.open(fetch(url)) as src:
    if not src.crs:
        raise ValueError('The LOLA GeoTIFF has no readable projection')
    width = height = 1024
    dem = physical(src, (height, width))
    affine = src.transform * rio.Affine.scale(src.width/width, src.height/height)
    dem.astype('<f4').tofile(out / 'south-pole-height.f32')
    sidecar(out / 'south-pole-height.json', source=url, credit='NASA GSFC PGDA / LOLA',
            radiusM=RADIUS, units='meters above reference sphere', width=width, height=height,
            encoding='float32-le', nodata='NaN', crsWkt=src.crs.to_wkt(), affine=list(affine)[:6],
            extentM=list(src.bounds), nativeResolutionM=80,
            rowOrder='north to south')
    # Sample the native grid in batches along great circles, including lunar curvature.
    from scipy.ndimage import map_coordinates
    from rasterio.windows import Window
    from rasterio.warp import transform as project
    distance = np.arange(80, max_range_m + 1, 80)
    angle = distance[None, :] / RADIUS
    bearing = np.deg2rad(azimuth[:, None])
    phi = np.deg2rad(lat)
    ray_lat = np.arcsin(np.sin(phi)*np.cos(angle) + np.cos(phi)*np.sin(angle)*np.cos(bearing))
    ray_lon = np.deg2rad(lon) + np.arctan2(np.sin(bearing)*np.sin(angle)*np.cos(phi),
                                          np.cos(angle)-np.sin(phi)*np.sin(ray_lat))
    ray_lon = (np.rad2deg(ray_lon) + 180) % 360 - 180
    xs, ys = project(GEO, src.crs, [lon] + ray_lon.ravel().tolist(),
                     [lat] + np.rad2deg(ray_lat).ravel().tolist())
    cols, rows = (~src.transform) * (np.asarray(xs), np.asarray(ys))
    left, top = max(0, int(cols.min())-2), max(0, int(rows.min())-2)
    right, bottom = min(src.width, int(cols.max())+3), min(src.height, int(rows.max())+3)
    window = Window(left, top, right-left, bottom-top)
    patch = src.read(1, window=window, out_dtype='float32', masked=True).filled(np.nan)
    patch = patch * src.scales[0] + src.offsets[0]
    sampled = map_coordinates(patch, [rows-top-0.5, cols-left-0.5],
                              order=1, mode='constant', cval=np.nan)
    if not np.isfinite(sampled).all():
        raise ValueError('Some sightlines leave valid DEM coverage. Reduce range or pick a site closer to the pole.')
site_radius = RADIUS + sampled[0] + eye_m
ray_radius = RADIUS + sampled[1:].reshape(len(azimuth), len(distance))
elevation = np.rad2deg(np.arctan2(ray_radius*np.cos(angle)-site_radius,
                                 ray_radius*np.sin(angle)))
horizon_deg = elevation.max(axis=1)
sidecar(out / 'south-pole-horizon.json', source=url, credit='NASA GSFC PGDA / LOLA',
        siteLatLon=[lat, lon], observerHeightM=eye_m, maxRangeM=max_range_m, distanceStepM=80,
        azimuthConvention='clockwise from geographic north',
        azimuthDeg=azimuth.tolist(), elevationDeg=horizon_deg.round(4).tolist(),
        note='Estimated horizon from the 80 m DEM out to the recorded range. Finer or more distant terrain can change it.')
pack(out)


## 3. South pole sunlight, Earth visibility and permanent shadow

Export [NASA PGDA product 69](https://pgda.gsfc.nasa.gov/products/69) as display PNGs and georeferenced numerical grids. Use these long-term maps as context alongside the date-specific JPL Horizons estimates. NASA counts Earth visibility when any of its disk is above the terrain.

In [ ]:
out = output_dir('south-pole-visibility')
base = 'https://pgda.gsfc.nasa.gov/data/MoonIllumination/'
products = [
    ('sun-average', 'AVGVISIB_85S_060M_201608.TIF', 'average solar illumination'),
    ('earth-average', 'AVGVISIB_85S_060M_201608_EARTH.TIF', 'any part of Earth disk visible'),
    ('permanent-shadow', 'LPSR_85S_060M_201608.TIF', 'permanent shadow classification'),
]
for key, filename, meaning in products:
    url = base + filename
    with rio.open(fetch(url)) as src:
        if not src.crs:
            raise ValueError(f'Missing projection on {filename}')
        size = 1024
        method = Resampling.nearest if key == 'permanent-shadow' else Resampling.average
        values = physical(src, (size, size), method)
        transform = src.transform * rio.Affine.scale(src.width/size, src.height/size)
        values.astype('<f4').tofile(out / f'{key}.f32')
        low, high = float(np.nanmin(values)), float(np.nanmax(values))
        image(values, low, high if high > low else low + 1, alpha=True).save(out / f'{key}.png')
        sidecar(out / f'{key}.json', source=url, credit='NASA GSFC PGDA / LOLA, Mazarico et al. (2011)',
                sourcePage='https://pgda.gsfc.nasa.gov/products/69', product=meaning,
                description='Long-term NASA context layer. PNG uses a contrast stretch for display.',
                units=src.units[0] or 'source GeoTIFF values with GDAL scale/offset applied',
                width=size, height=size, encoding='float32-le', nodata='NaN',
                affine=list(transform)[:6], extentM=list(src.bounds), crsWkt=src.crs.to_wkt(),
                dataRange=[low, high], nativeResolutionM=60, resampling=method.name)
pack(out)


## 4. Global normalized WAC texture

Build a 4096 × 2048 Moon texture from eight 64 px/degree equatorial tiles and two 200 m polar tiles. Source: [LROC 643 nm normalized reflectance](https://data.lroc.im-ldi.com/lroc/view_rdr/WAC_EMP).

In [ ]:
out = output_dir('normalized-wac')
width, height = 4096, 2048
# Build at 12 pixels/degree, so the 60° and 90° tile edges land on pixel boundaries.
ppd = 12
mosaic = np.full((180*ppd, 360*ppd), np.nan, dtype='float32')
sources = []
for hemisphere, row_start in [('N', 30*ppd), ('S', 90*ppd)]:
    for start_lon in range(0, 360, 90):
        name = f'WAC_EMP_643NM_E300{hemisphere}{(start_lon+45)*10:04d}_064P'
        url = f'{LROC}/DATA/MDR/WAC_EMP/{name}.IMG'
        with rio.open(fetch(url)) as src:
            if (src.width, src.height) != (90*64, 60*64):
                raise ValueError(f'Unexpected 64 px/deg grid for {name}')
            tile = physical(src, (60*ppd, 90*ppd))
        # Atlas equirectangular texture: west edge is −180°, center is 0°.
        column = ((start_lon+180) % 360)*ppd
        mosaic[row_start:row_start+60*ppd, column:column+90*ppd] = tile
        sources.append(url)
for hemisphere, row_start, south, north in [('N', 0, 60, 90), ('S', 150*ppd, -90, -60)]:
    name = f'WAC_EMP_643NM_P900{hemisphere}0000_152P'
    url = f'{LROC}/DATA/MDR/WAC_EMP/{name}.IMG'
    transform = from_bounds(-180, south, 180, north, 360*ppd, 30*ppd)
    cap = np.full((30*ppd, 360*ppd), np.nan, dtype='float32')
    with rio.open(fetch(url)) as src:
        if not src.crs:
            raise ValueError(f'Cannot read polar projection for {name}')
        reproject(rio.band(src, 1), cap, src_transform=src.transform, src_crs=src.crs,
                  src_nodata=src.nodata, dst_transform=transform, dst_crs=GEO,
                  dst_nodata=np.nan, resampling=Resampling.bilinear)
        cap *= src.scales[0]
        cap += src.offsets[0]
    mosaic[row_start:row_start+30*ppd] = cap
    sources.append(url)
# Reflectance is positive. Treat missing/nonpositive values as gaps and reject
# incomplete exports instead of silently painting large black tiles.
mosaic[mosaic <= 0] = np.nan
valid_fraction = float(np.isfinite(mosaic).mean())
if valid_fraction < 0.95:
    raise ValueError(f'Only {valid_fraction:.1%} of the globe is valid. Check source rasters before export.')
lo, hi = np.nanpercentile(mosaic, [1, 99])
globe = image(mosaic, lo, hi).resize((width, height), Image.Resampling.LANCZOS)
globe.save(out / 'wac-texture.jpg', quality=94)
sidecar(out / 'wac-texture.json', sources=sources, credit='NASA/GSFC/Arizona State University, LROC WAC',
        sourcePage='https://data.lroc.im-ldi.com/lroc/view_rdr/WAC_EMP', width=width, height=height,
        boundsDegrees=[-180, -90, 180, 90], rowOrder='north to south',
        projection='equirectangular', wavelengthNm=643, inputValidFraction=valid_fraction,
        displayStretch=[float(lo), float(hi)],
        note='Single global reflectance stretch. Remaining missing samples render black.')
display(globe.copy().resize((768, 384)))
pack(out)
